[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_computing/01_ieee754_floating_point_representation/exercises.ipynb)

# Exercises — Topic 01: IEEE 754 Floating-Point Representation

20 fully solved problems in 4 levels. Work each problem before opening its solution cell.

## Level 0 — Concept Check

### Problem L0.1: Epsilon versus the smallest float

State the difference between machine epsilon $\varepsilon_{\mathrm{mach}}$ and the smallest positive representable double, and give both values for binary64.

**Solution**

Machine epsilon is a measure of *relative precision*: the gap between $1$ and the next representable number,

$$
\varepsilon_{\mathrm{mach}} = 2^{-52} \approx 2.22 \times 10^{-16}
$$

The smallest positive double is a measure of *range*: the smallest subnormal,

$$
x_{\min} = 2^{e_{\min} - p} = 2^{-1022 - 52} = 2^{-1074} \approx 4.94 \times 10^{-324}
$$

(The smallest positive *normal* double is $2^{-1022} \approx 2.23 \times 10^{-308}$.) The two constants differ by roughly 300 orders of magnitude and answer different questions: "how finely can I resolve numbers near 1?" versus "how close to zero can I get?"

$$
\boxed{\varepsilon_{\mathrm{mach}} = 2^{-52} \approx 2.2 \times 10^{-16}, \qquad x_{\min} = 2^{-1074} \approx 4.9 \times 10^{-324}}
$$

*Key takeaway*: epsilon measures precision, the underflow threshold measures range — never confuse the two.

### Problem L0.2: Why `0.1 + 0.2 != 0.3`

Explain, in terms of representability, why the expression `0.1 + 0.2 == 0.3` evaluates to `False` in every IEEE 754 language.

**Solution**

A binary float is a dyadic rational $m \cdot 2^{e}$. A fraction $a/b$ in lowest terms is dyadic iff $b$ is a power of 2. The denominators of $1/10$, $2/10 = 1/5$, and $3/10$ all contain the prime factor 5, so none of the three decimals is representable; each is rounded to the nearest double at parse time.

The sum $\mathrm{fl}(\mathrm{fl}(0.1) + \mathrm{fl}(0.2))$ involves three roundings (two representations plus one addition). The accumulated error lands the left-hand side on the double $0.30000000000000004\dots$, while `0.3` parses to the *different* double $0.29999999999999998\dots$ — adjacent floats, one ulp apart.

$$
\boxed{\text{Both sides are (different) roundings of non-representable decimals, so equality fails by one ulp.}}
$$

*Key takeaway*: representation error occurs at parse time, before any arithmetic; compare floats with scale-aware tolerances.

### Problem L0.3: Special-value comparisons

For IEEE 754 doubles, state the truth value of each comparison and justify: (a) `NaN == NaN`, (b) `-0.0 == 0.0`, (c) `inf == inf`, (d) `inf - inf == 0`.

**Solution**

(a) **False.** NaN is *unordered*: every comparison involving NaN (except `!=`) returns false. This is by design, so that `x != x` detects NaN portably.

(b) **True.** IEEE 754 defines $-0$ and $+0$ as *equal* in comparisons, though they are distinct bit patterns (they differ under `1/x`: $1/{+0} = +\infty$ but $1/{-0} = -\infty$).

(c) **True.** $+\infty$ is a single well-defined value equal to itself; the ordering $-\infty \lt x \lt +\infty$ holds for all finite $x$.

(d) **False.** $\infty - \infty$ is an *invalid operation* producing NaN, and by (a) `NaN == 0` is false.

$$
\boxed{\text{(a) False, (b) True, (c) True, (d) False}}
$$

*Key takeaway*: the special values obey exact algebraic rules — learn them rather than avoiding them.

### Problem L0.4: The meaning of "correctly rounded"

Define what IEEE 754 means by a *correctly rounded* operation, and state the resulting error model for $z = \mathrm{fl}(x + y)$.

**Solution**

An operation is correctly rounded if the computed result equals the result of (i) computing the *exact* real-number answer, then (ii) rounding that single real number to the nearest representable float under the active rounding mode.

For round-to-nearest with unit roundoff $u$, this yields the standard model

$$
\mathrm{fl}(x + y) = (x + y)(1 + \delta), \qquad \lvert \delta \rvert \le u = 2^{-53} \text{ (binary64)}
$$

The same guarantee holds for $-, \times, \div, \sqrt{\phantom{x}}$. It does *not* generally hold for $\exp$, $\log$, $\sin$ in standard math libraries, which are typically only faithfully rounded (error $\le 1$ ulp).

$$
\boxed{\mathrm{fl}(x \circ y) = (x \circ y)(1+\delta), \quad \lvert \delta \rvert \le u}
$$

*Key takeaway*: one exact operation plus one rounding — this axiom is the foundation of all error analysis in Topics 02–05.

## Level 1 — Foundation

### Problem L1.1: Decoding a binary32 bit pattern

Decode the binary32 word with sign bit $s = 0$, exponent field $E = (10000000)_2$, and fraction field $f = (10000000000000000000000)_2$.

**Solution**

**Step 1 — exponent.** $E = (10000000)_2 = 128$. The binary32 bias is $2^{8-1} - 1 = 127$, so the unbiased exponent is $e = 128 - 127 = 1$.

**Step 2 — significand.** The leading 1 is implicit; the fraction $f$ has a single leading 1 bit, contributing $2^{-1}$:

$$
m = 1 + 2^{-1} = 1.5
$$

**Step 3 — assemble.**

$$
x = (-1)^0 \cdot 1.5 \cdot 2^{1} = 3.0
$$

$$
\boxed{x = 3.0}
$$

*Key takeaway*: value $= (-1)^s (1 + f/2^{23}) \cdot 2^{E - 127}$ — decoding is pure arithmetic once bias and hidden bit are recalled.

### Problem L1.2: Computing an ulp

Compute $\mathrm{ulp}(1000)$ in binary64, and verify it against the relative-precision band $\frac{1}{2}\varepsilon \lvert x \rvert \lt \mathrm{ulp}(x) \le \varepsilon \lvert x \rvert$.

**Solution**

**Step 1 — locate the binade.** $2^{9} = 512 \le 1000 \lt 1024 = 2^{10}$, so $e = 9$.

**Step 2 — apply the spacing formula.** With $p = 52$,

$$
\mathrm{ulp}(1000) = 2^{e - p} = 2^{9 - 52} = 2^{-43} \approx 1.137 \times 10^{-13}
$$

**Step 3 — check the band.** $\varepsilon \lvert x \rvert = 2^{-52} \cdot 1000 \approx 2.22 \times 10^{-13}$ and half of it is $\approx 1.11 \times 10^{-13}$. Indeed $1.11 \times 10^{-13} \lt 1.137 \times 10^{-13} \le 2.22 \times 10^{-13}$. ✓

$$
\boxed{\mathrm{ulp}(1000) = 2^{-43} \approx 1.14 \times 10^{-13}}
$$

*Key takeaway*: `np.spacing(1000.0)` returns exactly this; spacing is a step function that doubles at each power of two.

### Problem L1.3: The integer cliff at $2^{53}$

Show that $2^{53} + 1$ is not representable in binary64 and determine the value of $\mathrm{fl}(2^{53} + 1)$ under round-to-nearest-even.

**Solution**

**Step 1.** Numbers in the binade $[2^{53}, 2^{54})$ have spacing $2^{53 - 52} = 2$. The representable numbers there are exactly the *even* integers.

**Step 2.** $2^{53} + 1$ is odd, hence not representable. Its two neighbors are $2^{53}$ and $2^{53} + 2$, each at distance 1 — an exact tie.

**Step 3.** Round-to-nearest-*even* breaks the tie toward the neighbor with even significand: $2^{53}$ has significand $1.00\dots0$ (last bit 0, even), while $2^{53} + 2$ has last significand bit 1. Therefore

$$
\mathrm{fl}(2^{53} + 1) = 2^{53}
$$

$$
\boxed{\mathrm{fl}(2^{53} + 1) = 2^{53} = 9007199254740992}
$$

*Key takeaway*: above $2^{53}$, doubles skip integers — never store large IDs, counters, or database keys in floats.

### Problem L1.4: Encoding $1.0$ in binary32

What are the stored sign, exponent, and fraction fields of the binary32 representation of $1.0$? Give the exponent field in binary.

**Solution**

Write $1.0 = (-1)^0 \cdot (1.000\dots0)_2 \cdot 2^{0}$.

- Sign: $s = 0$.
- Unbiased exponent $e = 0$, so the stored field is $E = e + 127 = 127 = (01111111)_2$.
- Fraction: all 23 bits zero (the leading 1 is implicit).

The full word is $0\,\vert\,01111111\,\vert\,00000000000000000000000$, i.e. hexadecimal `0x3F800000`.

$$
\boxed{s = 0, \quad E = (01111111)_2 = 127, \quad f = 0 \quad (\texttt{0x3F800000})}
$$

*Key takeaway*: the bias makes exponent comparison work with unsigned integer comparison — a deliberate hardware-friendly design choice.

### Problem L1.5: Precision of fp16

For IEEE binary16 (fp16), compute machine epsilon, unit roundoff, and the approximate number of significant decimal digits.

**Solution**

fp16 has $p = 10$ fraction bits (11-bit significand with the hidden bit).

**Machine epsilon:**

$$
\varepsilon_{\mathrm{mach}} = 2^{-10} = \frac{1}{1024} \approx 9.77 \times 10^{-4}
$$

**Unit roundoff:**

$$
u = 2^{-11} \approx 4.88 \times 10^{-4}
$$

**Decimal digits:**

$$
d = 11 \log_{10} 2 \approx 11 \times 0.30103 \approx 3.31
$$

$$
\boxed{\varepsilon = 2^{-10} \approx 9.8 \times 10^{-4}, \quad u = 2^{-11} \approx 4.9 \times 10^{-4}, \quad d \approx 3.3 \text{ digits}}
$$

*Key takeaway*: fp16 resolves about 1 part in 2000 — adequate for activations and gradients under SGD noise, hopeless for accumulating averages (use fp32 accumulators).

### Problem L1.6: Applying the Sterbenz lemma

Using the Sterbenz lemma, decide whether each subtraction is exact in binary64: (a) $1.75 - 1.5$, (b) $100 - 0.5$, (c) $2^{-1022} \cdot 1.5 - 2^{-1022}$ (deep in the normal range boundary).

**Solution**

The Sterbenz lemma guarantees $\mathrm{fl}(x - y) = x - y$ whenever $y/2 \le x \le 2y$ (both representable).

(a) $x = 1.75$, $y = 1.5$: check $0.75 \le 1.75 \le 3.0$. ✓ Exact; indeed $0.25 = 2^{-2}$ is representable.

(b) $x = 100$, $y = 0.5$: $2y = 1 \lt 100$, condition fails. The lemma gives no guarantee — though here $99.5$ happens to be representable anyway (spacing near 100 is $2^{-46}$, and $99.5$ is a dyadic rational). The lemma is sufficient, not necessary.

(c) $x = 1.5 \cdot 2^{-1022}$, $y = 2^{-1022}$: $y/2 \le x \le 2y$ holds. The difference $0.5 \cdot 2^{-1022} = 2^{-1023}$ is *subnormal*, and the lemma still applies because gradual underflow makes the result representable. Exact. ✓

$$
\boxed{\text{(a) exact by Sterbenz; (b) no guarantee (though exact here); (c) exact — subnormals rescue the result}}
$$

*Key takeaway*: nearby subtraction commits no new error; without subnormals, case (c) would flush to zero and break the lemma.

## Level 2 — Applications in AI/ML

### Problem L2.1: fp16 overflow in attention logits

In scaled dot-product attention, logits are $z = q^{\top} k / \sqrt{d}$ with $q, k \in \mathbb{R}^{d}$. Suppose entries of $q$ and $k$ are i.i.d. with mean 0 and variance $\sigma^2$. (a) Compute $\mathrm{Var}(q^{\top} k)$ and the typical magnitude of the *unscaled* logit for $d = 4096$, $\sigma = 4$. (b) Explain why the computation overflows in fp16 without the $1/\sqrt{d}$ scaling, given that the fp16 maximum is 65504.

**Solution**

**(a)** For independent zero-mean entries, $\mathrm{Var}(q_i k_i) = \mathbb{E}[q_i^2 k_i^2] = \sigma^4$, and the $d$ terms are uncorrelated:

$$
\mathrm{Var}\left(q^{\top} k\right) = d \sigma^4 = 4096 \cdot 256 = 1048576
$$

Typical magnitude $= \sqrt{d}\,\sigma^2 = 64 \cdot 16 = 1024$; three-sigma excursions reach $\approx 3072$.

**(b)** The unscaled dot product itself is within fp16 range here, but the *intermediate accumulations* and the subsequent $e^{z}$ in softmax are not: $e^{1024}$ overflows any format (fp16 overflows already at $z \gt \ln 65504 \approx 11.09$). With the $1/\sqrt{d}$ scaling the logit has standard deviation $\sigma^2 = 16$, still risky for fp16 exponentials — which is why softmax is computed with the max-subtraction trick (Topic 05) and accumulations run in fp32.

$$
\boxed{\mathrm{Var}(q^{\top}k) = d\sigma^4; \quad \text{typical } \lvert z_{\text{unscaled}} \rvert \approx \sqrt{d}\sigma^2 = 1024 \gg \ln(65504) \approx 11.1}
$$

*Key takeaway*: the $1/\sqrt{d}$ in attention is a variance-control device that doubles as overflow protection for low-precision formats.

### Problem L2.2: Gradient underflow in fp16

A gradient component has magnitude $g = 10^{-6}$. (a) Classify $g$ in fp16 (normal, subnormal, or underflow to zero). (b) How many significant bits survive? (c) What loss-scale factor $2^{k}$ moves $g$ into the normal range?

**Solution**

**(a)** fp16 minimum normal is $2^{-14} \approx 6.10 \times 10^{-5}$; smallest subnormal is $2^{-24} \approx 5.96 \times 10^{-8}$. Since $5.96 \times 10^{-8} \lt 10^{-6} \lt 6.10 \times 10^{-5}$, the value is **subnormal** — representable but with reduced precision.

**(b)** Subnormals have the form $(0.b_1 \dots b_{10})_2 \cdot 2^{-14}$ with absolute spacing $2^{-24}$. The number of significant bits of $g$ is

$$
\log_2 \frac{g}{2^{-24}} = \log_2 \left(10^{-6} \cdot 2^{24}\right) \approx \log_2 16.8 \approx 4.07
$$

so only about **4 bits** ($\approx 1.2$ decimal digits) survive, versus 11 for a normal value.

**(c)** We need $2^{k} g \ge 2^{-14}$, i.e. $2^{k} \ge 2^{-14}/10^{-6} \approx 16.4$, so $k = 5$ (scale 32) suffices; practical systems use $k \approx 15$ (scale $2^{15} = 32768$) to protect much smaller gradients too.

$$
\boxed{g = 10^{-6} \text{ is subnormal in fp16 (} \approx 4 \text{ bits); loss scale } 2^{5} \text{ normalizes it}}
$$

*Key takeaway*: fp16 gradients do not just vanish abruptly — they first pass through a zone of silent precision decay; loss scaling shifts the whole distribution out of it.

### Problem L2.3: bfloat16 versus fp16 rounding error

Round $x = 1.01$ to (a) fp16 and (b) bfloat16, giving the relative error in each case. Which format represents this *particular* value better, and why?

**Solution**

Both formats have $x$ in the binade $[1, 2)$, where the spacing is $\varepsilon_{\mathrm{mach}}$.

**(a) fp16**: spacing $2^{-10} = 1/1024$. We need the nearest multiple of $1/1024$ to $0.01$: $0.01 \times 1024 = 10.24$, so the fraction rounds to $10/1024$ and

$$
\mathrm{fl}_{16}(1.01) = 1 + \frac{10}{1024} = 1.009765625, \qquad \text{rel. err.} = \frac{0.000234375}{1.01} \approx 2.3 \times 10^{-4}
$$

**(b) bfloat16**: spacing $2^{-7} = 1/128$. $0.01 \times 128 = 1.28$, so the fraction rounds to $1/128$ and

$$
\mathrm{fl}_{bf16}(1.01) = 1 + \frac{1}{128} = 1.0078125, \qquad \text{rel. err.} = \frac{0.0021875}{1.01} \approx 2.2 \times 10^{-3}
$$

fp16 is about $9.5\times$ more accurate here, consistent with its 3 extra fraction bits ($2^{3} = 8\times$ finer grid).

$$
\boxed{\text{fp16: rel. err. } \approx 2.3 \times 10^{-4}; \quad \text{bf16: rel. err. } \approx 2.2 \times 10^{-3}}
$$

*Key takeaway*: per-value, fp16 always beats bf16 by $\approx 8\times$ in precision; bf16 wins only when *range* (not precision) is the binding constraint.

### Problem L2.4: Drift in accumulated time steps

A simulation advances time by repeatedly adding the double $\mathrm{fl}(0.1)$, whose representation error is $+\delta_0$ with $\delta_0 \approx 5.55 \times 10^{-18}$. Ignoring the (smaller, oscillating) addition rounding, estimate the accumulated time error after $10^{9}$ steps, and state the correct implementation pattern.

**Solution**

Each step adds the same signed bias $\delta_0$, so errors accumulate *linearly and coherently* (no cancellation):

$$
E_n \approx n \delta_0 = 10^{9} \times 5.55 \times 10^{-18} \approx 5.6 \times 10^{-9} \text{ s}
$$

That looks small, but continuing to $n = 10^{13}$ steps (a long molecular-dynamics run) gives $\approx 5.6 \times 10^{-5}$ s of drift, and in single precision the same pattern produces error $\approx n \times 10^{-9}$ — seconds of drift within hours (the Patriot failure mode).

**Correct pattern**: never accumulate; recompute from an integer counter,

$$
t_n = n \cdot \mathrm{fl}(\Delta t)
$$

which carries a *single* rounding ($\lvert \text{err} \rvert \le u \cdot t_n$, relative not absolute growth) because integers $n \lt 2^{53}$ and one multiplication are exact-to-one-rounding.

$$
\boxed{E_n \approx n \delta_0 \approx 5.6 \times 10^{-9} \text{ s after } 10^{9} \text{ steps; use } t = n \cdot \Delta t \text{ instead}}
$$

*Key takeaway*: a biased representation error, added $n$ times, grows like $n$; one multiplication commits it only once.

### Problem L2.5: Embedding IDs in float32

A recommender system stores item IDs in a float32 tensor. Show that ID $16777217$ is corrupted, and find the largest ID that is guaranteed safe.

**Solution**

float32 has $p = 23$ fraction bits, so all integers up to $2^{24} = 16777216$ are exactly representable. In the binade $[2^{24}, 2^{25})$ the spacing is $2^{24-23} = 2$: only even integers exist there.

$16777217 = 2^{24} + 1$ is odd, so it rounds (ties-to-even) to $2^{24} = 16777216$ — a *different* item. Every odd ID above $2^{24}$ collides with a neighbor.

$$
\boxed{\text{Largest safe integer in float32} = 2^{24} = 16777216}
$$

*Key takeaway*: catalogs beyond ~16.8M items cannot use float32 IDs; keep IDs in integer dtypes and convert only embeddings, never keys, to floating point.

### Problem L2.6: Designing a tolerance for `allclose`

A unit test compares a layer's output tensor $y$ (float32, values of order $10^{2}$) against a reference. The computation chains roughly $N = 10^{4}$ flops per output element. Propose principled `rtol` and `atol` values for `np.allclose` and justify them.

**Solution**

**Relative tolerance.** Worst-case accumulated relative error after $N$ roundings is of order $N u$; the *expected* error under random-walk cancellation is of order $\sqrt{N} u$. With $u_{32} = 2^{-24} \approx 6 \times 10^{-8}$:

$$
\sqrt{N} u = 10^{2} \times 6 \times 10^{-8} = 6 \times 10^{-6}
$$

A safety factor of ~10 gives $\mathrm{rtol} \approx 10^{-4}$ to $10^{-5}$. The NumPy default $\mathrm{rtol} = 10^{-5}$ sits exactly in this band for shallow computations.

**Absolute tolerance.** `atol` governs comparisons where the reference is near zero. Outputs of order $10^{2}$ that cancel to near zero can retain absolute error $\approx 10^{2} \times \sqrt{N} u \approx 6 \times 10^{-4}$; choose $\mathrm{atol} \approx 10^{-3}$ if exact zeros are expected, or set $\mathrm{atol} = 0$ to *force* purely relative comparison when outputs are bounded away from zero.

$$
\boxed{\mathrm{rtol} \approx 10 \sqrt{N}\,u \approx 10^{-4}, \qquad \mathrm{atol} \approx \lvert y \rvert_{\max} \sqrt{N}\,u \text{ or } 0}
$$

*Key takeaway*: tolerances are derived quantities — from precision $u$, depth $N$, and data scale — not folklore constants.

## Level 3 — Challenge

### Problem L3.1: Proving the Sterbenz lemma

Prove: if $x, y$ are positive floats in $\mathbb{F}(p, e_{\min}, e_{\max})$ with $y/2 \le x \le 2y$, then $x - y \in \mathbb{F}$ (assuming gradual underflow).

**Solution**

WLOG assume $y \le x \le 2y$ (the case $x \lt y$ is symmetric via $\lvert x - y \rvert$).

**Step 1 — integer scaling.** Write $x = M_x 2^{e_x - p}$ and $y = M_y 2^{e_y - p}$ with integer significands $M_x, M_y \in [2^{p}, 2^{p+1})$ (normal case). From $y \le x \le 2y$ we get $e_y \le e_x \le e_y + 1$.

**Step 2 — common exponent.** Rescale both to the smaller exponent $e_y$: define $M_x' = M_x 2^{e_x - e_y} \in \{M_x, 2M_x\}$, an integer $\lt 2^{p+2}$. Then

$$
x - y = (M_x' - M_y)\, 2^{e_y - p}
$$

**Step 3 — bound the integer difference.** Since $x \le 2y$,

$$
M_x' - M_y = \frac{x - y}{2^{e_y - p}} \le \frac{y}{2^{e_y - p}} = M_y \lt 2^{p+1}
$$

and $M_x' - M_y \ge 0$. So $x - y$ is an integer multiple $k \cdot 2^{e_y - p}$ with $0 \le k \lt 2^{p+1}$, i.e. it needs at most $p + 1$ significand bits at exponent scale $e_y$ — exactly what the format provides (with subnormals covering the case where normalization would push the exponent below $e_{\min}$). Hence $x - y$ is representable and the correctly rounded subtraction returns it exactly. $\blacksquare$

$$
\boxed{y/2 \le x \le 2y \implies \mathrm{fl}(x - y) = x - y}
$$

*Key takeaway*: the proof is bookkeeping on integer significands — the difference simply never needs more bits than the operands had.

### Problem L3.2: Counting all fp16 values

Count exactly how many distinct finite real values binary16 represents (count $\pm 0$ as two values), and how many bit patterns are NaN.

**Solution**

fp16 layout: 1 sign bit, $k = 5$ exponent bits ($E \in \{0, \dots, 31\}$), 10 fraction bits.

**Normals** ($1 \le E \le 30$): each of the 30 exponent values admits $2^{10} = 1024$ fractions, times 2 signs:

$$
N_{\text{normal}} = 2 \times 30 \times 1024 = 61440
$$

**Subnormals and zeros** ($E = 0$): $1024$ fractions (including $f = 0$, which gives $\pm 0$), times 2 signs:

$$
N_{\text{sub+zero}} = 2 \times 1024 = 2048
$$

**Finite total**: $61440 + 2048 = 63488$. All are distinct reals except that $+0$ and $-0$ coincide as reals — counting signed zeros separately as instructed, the answer stands at 63488 patterns representing $63487$ distinct reals.

**Non-finite** ($E = 31$): $f = 0$ gives $\pm\infty$ (2 patterns); the remaining $2 \times 1023 = 2046$ patterns are NaN.

Check: $63488 + 2 + 2046 = 65536 = 2^{16}$. ✓

$$
\boxed{63488 \text{ finite patterns } (61440 \text{ normal}, 2048 \text{ subnormal/zero}); \quad 2046 \text{ NaN patterns}}
$$

*Key takeaway*: 3% of all fp16 bit patterns are NaN — a nontrivial slice of the format is spent on error signaling.

### Problem L3.3: The wobble of relative precision

Show that for round-to-nearest in a binary format, the worst-case relative representation error at $x$ oscillates ("wobbles") between $u/2$ and $u$ across each binade — i.e. relative precision is only defined up to a factor of 2. Where in the binade is accuracy best?

**Solution**

Let $x \in [2^{e}, 2^{e+1})$. The absolute rounding error is bounded by half the local spacing, a *constant* within the binade:

$$
\lvert \mathrm{fl}(x) - x \rvert \le \tfrac{1}{2} \cdot 2^{e-p}
$$

The worst-case *relative* error at $x$ is therefore

$$
r(x) = \frac{2^{e-p-1}}{x}
$$

which is a decreasing function of $x$ across the binade:

- At the left edge $x = 2^{e}$: $r = 2^{-p-1} = u$ (worst).
- At the right edge $x \to 2^{e+1}$: $r \to 2^{-p-2} = u/2$ (best).

So the guaranteed relative accuracy improves by a factor of 2 as the significand fills up from $1.000\dots$ toward $1.111\dots$, then snaps back to $u$ when the exponent increments. This factor-2 oscillation is Goldberg's "wobble"; it is why careful analyses (and the definition of precision in decimal formats) quote worst-case $u$ rather than a single sharp constant.

$$
\boxed{r(x) = \frac{2^{e-p-1}}{x} \in \left(\frac{u}{2},\, u\right], \text{ worst just above powers of } 2, \text{ best just below them}}
$$

*Key takeaway*: relative precision is not a single number but a factor-2 band — algorithms sensitive to the last bit must be analyzed against the pessimistic edge.

### Problem L3.4: Stochastic rounding is unbiased

Let $x$ lie between adjacent floats $a \lt x \lt b$ with gap $h = b - a$. Stochastic rounding sets $\mathrm{SR}(x) = b$ with probability $q = (x - a)/h$ and $\mathrm{SR}(x) = a$ otherwise. (a) Prove $\mathbb{E}[\mathrm{SR}(x)] = x$. (b) Compute $\mathrm{Var}(\mathrm{SR}(x))$ and its maximum over $x$. (c) Explain why unbiasedness matters for training with tiny learning rates.

**Solution**

**(a)**

$$
\mathbb{E}[\mathrm{SR}(x)] = a(1 - q) + b q = a + (b - a)\frac{x - a}{h} = a + (x - a) = x \qquad \blacksquare
$$

**(b)** With $t = x - a \in (0, h)$:

$$
\mathrm{Var}(\mathrm{SR}(x)) = \mathbb{E}[\mathrm{SR}(x)^2] - x^2 = (b - x)(x - a) = t(h - t)
$$

maximized at the midpoint $t = h/2$ with value $h^2/4$. So one stochastic rounding has standard deviation up to $h/2$ — *twice* the worst-case error of round-to-nearest — but zero mean.

**(c)** With round-to-nearest, a weight update $\lvert \eta g \rvert \lt \frac{1}{2}\mathrm{ulp}(w)$ rounds to *zero change* deterministically, every step: learning stalls even though $n$ steps should move the weight by $n \eta g$. Under stochastic rounding each step moves the weight by one ulp with probability $\eta \lvert g \rvert / h$, so the *expected* trajectory matches exact arithmetic: $\mathbb{E}[\Delta w_n] = n \eta g$. Bias, not variance, is the enemy of accumulation — SGD already tolerates variance.

$$
\boxed{\mathbb{E}[\mathrm{SR}(x)] = x, \quad \mathrm{Var} = (b - x)(x - a) \le \frac{h^2}{4}}
$$

*Key takeaway*: stochastic rounding trades a larger random error for a zero systematic one — exactly the right trade for long accumulations in low precision.